In [2]:
# Tools for math, tables, and file paths
import numpy as np
import pandas as pd
from pathlib import Path

# Your feature builder from Phase 2
from sitebench.features import SiteFeatures

# Find the repo root (the folder with .git) so file paths work wherever the notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# Load the training and validation sites; test stays locked until Phase 5
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv", low_memory=False)
val = pd.read_csv(ROOT / "data" / "processed" / "val.csv", low_memory=False)

# Show the TRIR at several percentiles of the training sites
for p in [50, 60, 75, 90, 95]:
    # The TRIR below which p% of training sites fall
    print(f"{p}th percentile: TRIR {np.percentile(train['trir'], p):.2f}")

50th percentile: TRIR 1.67
60th percentile: TRIR 3.10
75th percentile: TRIR 5.68
90th percentile: TRIR 11.28
95th percentile: TRIR 16.91


In [3]:
# Learn the cutoffs from training data only: the TRIR at the 50th and 90th percentiles
LOW_CUTOFF = np.percentile(train["trir"], 50)
HIGH_CUTOFF = np.percentile(train["trir"], 90)


def risk_level(trir):
    """Label each site low, medium, or high using the training cutoffs."""
    # np.select checks the conditions in order and uses the first one that's true;
    # anything that fails both gets the default, "high"
    return np.select(
        [trir < LOW_CUTOFF, trir < HIGH_CUTOFF],
        ["low", "medium"],
        default="high",
    )


# Add a risk_level column to both datasets
train["risk_level"] = risk_level(train["trir"])
val["risk_level"] = risk_level(val["trir"])

# Show what share of sites landed in each level (normalize=True gives fractions instead of counts)
print("Train:", train["risk_level"].value_counts(normalize=True).round(3).to_dict())
print("Val:  ", val["risk_level"].value_counts(normalize=True).round(3).to_dict())

Train: {'low': 0.5, 'medium': 0.4, 'high': 0.1}
Val:   {'low': 0.497, 'medium': 0.402, 'high': 0.101}


In [5]:
def gini(labels):
    """How mixed a group is: 0 = all one risk level, higher = more mixed."""
    # np.unique finds each distinct risk level and counts how many sites have it
    _, counts = np.unique(labels, return_counts=True)
    # Turn counts into shares, like 4 of 5 sites -> 0.8
    shares = counts / len(labels)
    # TODO: 1 minus the sum of the squared shares
    return (1 - np.sum(shares ** 2))


# The three groups you worked by hand
print(gini(["L", "L", "L", "L", "L", "M", "M", "M", "M", "H"]))  # the full toy set
print(gini(["M", "M", "M", "M", "H"]))                            # Question B's yes group
print(gini(["L", "L", "L", "L", "L"]))                            # Question B's no group

0.58
0.31999999999999984
0.0


In [7]:
def split_score(values, labels, threshold):
    """Weighted Gini after asking 'is this feature above the threshold?' Lower is better."""
    # Convert to NumPy arrays so we can filter them with True/False lists
    values = np.asarray(values)
    labels = np.asarray(labels)
    # True for every site that answers "yes" (above the threshold), False otherwise
    yes = values > threshold
    # Pick out each group's labels; ~yes flips True and False to get the "no" sites
    yes_labels = labels[yes]
    no_labels = labels[~yes]
    # Each group's share of all the sites, used as its weight
    yes_share = len(yes_labels) / len(labels)
    no_share = len(no_labels) / len(labels)
    # TODO: each group's share times its Gini, added together
    return yes_share * gini(yes_labels) + no_share * gini(no_labels)


# Toy data: the same 10 sites, now with hours per employee for each
hours = [1500, 1700, 1800, 1900, 2000, 2300, 2400, 2500, 2600, 2800]
levels = ["L", "L", "L", "L", "L", "M", "M", "M", "M", "H"]

print(split_score(hours, levels, 2200))
print(split_score(hours, levels, 1850))

0.15999999999999992
0.39999999999999997


In [11]:
def best_threshold(values, labels):
    """Try every midpoint between neighboring values; return the best cutoff and its score."""
    # np.unique sorts the distinct values, so neighbors sit next to each other
    unique_values = np.unique(values)
    # [:-1] is every value except the last, [1:] every value except the first,
    # so adding them pairs each value with its next neighbor; halving gives the midpoints
    candidates = (unique_values[:-1] + unique_values[1:]) / 2
    # No best cutoff yet; starting the best score at infinity means any real score beats it
    best_cut, best_score = None, np.inf
    for threshold in candidates:
        # TODO: score this cutoff using your split_score function
        score = split_score(values, labels, threshold)
        # If it beats the best so far, remember this cutoff and its score
        if score < best_score:
            best_cut, best_score = threshold, score
    return best_cut, best_score


print(best_threshold(hours, levels))

(np.float64(2150.0), np.float64(0.15999999999999992))


In [13]:
# Tables of features need pandas
import pandas as pd

# Toy data: 10 sites, two features each
toy_X = pd.DataFrame({
    "hours_per_employee": [1500, 1700, 1800, 1900, 2000, 2300, 2400, 2450, 2500, 2600],
    "is_mail_delivery":   [0,    0,    0,    0,    0,    0,    0,    1,    0,    0],
})
toy_levels = ["L", "L", "L", "L", "L", "M", "M", "H", "M", "M"]


def best_split(X, labels):
    """Try every feature and every cutoff; return the best feature, cutoff, and score."""
    # No best yet: no feature, no cutoff, and a score of infinity that anything beats
    best_feature, best_cut, best_score = None, None, np.inf
    # Repeat for each column name in the feature table
    for feature in X.columns:
        # TODO: find the best cutoff for this one feature, using best_threshold.
        # Its inputs are this feature's column of values, X[feature], and the labels
        cut, score = best_threshold(X[feature], labels)
        # If this feature's best beats the best so far, remember it
        if score < best_score:
            best_feature, best_cut, best_score = feature, cut, score
    return best_feature, best_cut, best_score


print(best_split(toy_X, toy_levels))

('hours_per_employee', np.float64(2150.0), np.float64(0.15999999999999992))


In [15]:
def majority(labels):
    """The most common risk level in a group: what a leaf answers."""
    # Count how many sites have each level, then pick the level with the biggest count
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


def build_tree(X, labels, depth=0, max_depth=2):
    """Grow a tree: find the best question, split, then grow a smaller tree on each side."""
    labels = np.asarray(labels)
    # Stop and make a leaf if the group is already pure or the tree is as deep as allowed
    if gini(labels) == 0 or depth == max_depth:
        return majority(labels)
    # Find the best question for this group of sites
    feature, cut, score = best_split(X, labels)
    # If no question can split the group (every feature value identical), make a leaf
    if feature is None:
        return majority(labels)
    # True for each site that answers yes to the question
    yes = (X[feature] > cut).to_numpy()
    # A question node: the question, plus a whole subtree for each answer
    return {
        "question": f"{feature} > {cut}",
        "feature": feature,
        "cut": cut,
        # Grow a tree on only the yes sites, one level deeper
        "yes": build_tree(X[yes], labels[yes], depth + 1, max_depth),
        # TODO: same thing for the no sites (~yes flips True and False)
        "no": build_tree(X[~yes], labels[~yes], depth + 1, max_depth),
    }


def print_tree(node, indent=""):
    """Show the tree as indented if/else rules."""
    # A leaf is just a risk level, so print it and stop
    if not isinstance(node, dict):
        print(indent + "-> " + node)
        return
    # A question node: print the question, then each side's subtree, indented further
    print(indent + "if " + node["question"] + ":")
    print_tree(node["yes"], indent + "    ")
    print(indent + "else:")
    print_tree(node["no"], indent + "    ")


toy_tree = build_tree(toy_X, toy_levels, max_depth=2)
print_tree(toy_tree)

if hours_per_employee > 2150.0:
    if is_mail_delivery > 0.5:
        -> H
    else:
        -> M
else:
    -> L
